# Análise Global de Mudanças Climáticas e Eventos Extremos
### Sistemas Distribuídos — Apache Spark

**Dataset Principal:** GlobalLandTemperaturesByCity.csv (Berkeley Earth / Kaggle)  
**Dataset Secundário:** owid-co2-data.csv (Our World in Data)

---

## 0. Configuração da SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, DateType
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import time

spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("ClimateAnalysis") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.cores", "2") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Spark UI: http://localhost:4040")

---
## 1. Ingestão e Limpeza dos Dados (ETL)

Seguindo o fluxo recomendado: Leitura → Limpeza → Enriquecimento → Processamento → Saída.

In [ ]:
# =============================================================
# BLOCO DE INGESTÃO: Leitura dos CSVs brutos para DataFrames
# =============================================================

# Dataset 1: Temperaturas
df_raw = spark.read.csv(
    "/home/jovyan/data/GlobalLandTemperaturesByCity.csv",
    header=True,
    inferSchema=True
)

print("=== Schema Original ===")
df_raw.printSchema()
print(f"Total de linhas brutas: {df_raw.count():,}")

In [ ]:
# =============================================================
# BLOCO DE TRANSFORMAÇÃO: Limpeza e Enriquecimento
# =============================================================

# A. Remover nulos na temperatura (principal coluna analítica)
df_clean = df_raw.dropna(subset=["AverageTemperature"])

# B. Conversão de tipos e extração de Ano e Mês
df_clean = df_clean \
    .withColumn("dt", F.to_date(F.col("dt"), "yyyy-MM-dd")) \
    .withColumn("Year", F.year(F.col("dt"))) \
    .withColumn("Month", F.month(F.col("dt"))) \
    .withColumn("Decade", (F.year(F.col("dt")) / 10).cast(IntegerType()) * 10)

# C. Garantir tipos numéricos corretos
df_clean = df_clean \
    .withColumn("AverageTemperature", F.col("AverageTemperature").cast(DoubleType())) \
    .withColumn("AverageTemperatureUncertainty", F.col("AverageTemperatureUncertainty").cast(DoubleType()))

# D. Filtro de Incerteza: remover medições com incerteza > 1.5°C
df_clean = df_clean.filter(F.col("AverageTemperatureUncertainty") < 1.5)

# E. Filtro de período histórico relevante (> 1900)
df_clean = df_clean.filter(F.col("Year") >= 1900)

# F. Limpeza de coordenadas (N/S/E/W → numérico)
df_clean = df_clean \
    .withColumn("Lat",
        F.when(F.col("Latitude").endswith("N"),
               F.regexp_replace(F.col("Latitude"), "N", "").cast(DoubleType()))
        .otherwise(-F.regexp_replace(F.col("Latitude"), "S", "").cast(DoubleType()))
    ) \
    .withColumn("Lon",
        F.when(F.col("Longitude").endswith("E"),
               F.regexp_replace(F.col("Longitude"), "E", "").cast(DoubleType()))
        .otherwise(-F.regexp_replace(F.col("Longitude"), "W", "").cast(DoubleType()))
    )

# G. Normalização de strings
df_clean = df_clean \
    .withColumn("Country", F.trim(F.col("Country"))) \
    .withColumn("City", F.trim(F.col("City")))

# Cache após limpeza — evita reprocessamento nas 8 consultas seguintes
df_clean.cache()

print(f"Linhas após limpeza: {df_clean.count():,}")
df_clean.show(5)

---
## Pergunta 1 — Média Móvel de Temperatura: Evolução da temperatura média anual global por década

In [ ]:
# =====================================================
# CÓDIGO RESPOSTA — PERGUNTA 1: Média por Década
# =====================================================

# Sem cache (primeira execução)
t0 = time.time()
df_decade = df_clean \
    .groupBy("Decade") \
    .agg(F.round(F.avg("AverageTemperature"), 2).alias("AvgTemp")) \
    .orderBy("Decade")
df_decade.show()
t1 = time.time()
print(f"Tempo (com cache ativo): {t1-t0:.2f}s")

# Visualização
pdf_decade = df_decade.toPandas()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(pdf_decade["Decade"], pdf_decade["AvgTemp"], marker="o", linewidth=2, color="#E74C3C")
ax.fill_between(pdf_decade["Decade"], pdf_decade["AvgTemp"],
                alpha=0.15, color="#E74C3C")
ax.set_title("Temperatura Média Global por Década (1900–2010s)", fontsize=14, fontweight="bold")
ax.set_xlabel("Década")
ax.set_ylabel("Temperatura Média (°C)")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("/home/jovyan/output/p1_temperatura_por_decada.png", dpi=150)
plt.show()
print("Gráfico salvo em /output/p1_temperatura_por_decada.png")

---
## Pergunta 2 — Anomalias Locais: 10 anos mais quentes por continente (últimos 50 anos)

In [ ]:
# =====================================================
# CÓDIGO RESPOSTA — PERGUNTA 2: Anos mais quentes por continente
# =====================================================

# Mapeamento país → continente (amostra representativa)
continent_map = {
    "Brazil": "América do Sul", "Argentina": "América do Sul", "Colombia": "América do Sul",
    "Peru": "América do Sul", "Chile": "América do Sul", "Venezuela": "América do Sul",
    "United States": "América do Norte", "Canada": "América do Norte", "Mexico": "América do Norte",
    "Germany": "Europa", "France": "Europa", "United Kingdom": "Europa",
    "Italy": "Europa", "Spain": "Europa", "Russia": "Europa",
    "China": "Ásia", "India": "Ásia", "Japan": "Ásia",
    "Indonesia": "Ásia", "Pakistan": "Ásia", "Bangladesh": "Ásia",
    "Nigeria": "África", "Ethiopia": "África", "Egypt": "África",
    "South Africa": "África", "Kenya": "África", "Tanzania": "África",
    "Australia": "Oceania", "New Zealand": "Oceania"
}

# Criar DataFrame de mapeamento no Spark
continent_rows = [(k, v) for k, v in continent_map.items()]
df_continent_map = spark.createDataFrame(continent_rows, ["Country", "Continent"])

# Filtrar últimos 50 anos
df_50y = df_clean.filter(F.col("Year") >= 1975)

# Join com mapeamento de continentes
df_cont = df_50y.join(df_continent_map, on="Country", how="inner")

# Temperatura média anual por continente/ano
df_cont_year = df_cont \
    .groupBy("Continent", "Year") \
    .agg(F.round(F.avg("AverageTemperature"), 2).alias("AvgTemp"))

# Window Function: rank dos anos mais quentes por continente
window_cont = Window.partitionBy("Continent").orderBy(F.desc("AvgTemp"))

df_top10_cont = df_cont_year \
    .withColumn("Rank", F.rank().over(window_cont)) \
    .filter(F.col("Rank") <= 10) \
    .orderBy("Continent", "Rank")

df_top10_cont.show(40, truncate=False)

---
## Pergunta 3 — Cidades em Risco: maior desvio padrão de temperatura (último século)

In [ ]:
# =====================================================
# CÓDIGO RESPOSTA — PERGUNTA 3: Cidades com maior instabilidade térmica
# =====================================================

df_stddev = df_clean \
    .filter(F.col("Year") >= 1923) \
    .groupBy("City", "Country") \
    .agg(
        F.round(F.stddev("AverageTemperature"), 4).alias("TempStdDev"),
        F.count("*").alias("NumRecords")
    ) \
    .filter(F.col("NumRecords") > 100) \
    .orderBy(F.desc("TempStdDev")) \
    .limit(20)

df_stddev.show(truncate=False)

# Visualização — Barras horizontais
pdf_std = df_stddev.toPandas()
pdf_std["Label"] = pdf_std["City"] + ", " + pdf_std["Country"]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(pdf_std["Label"][:15][::-1],
               pdf_std["TempStdDev"][:15][::-1],
               color="#2980B9", edgecolor="white")
ax.set_title("Top 15 Cidades com Maior Instabilidade Térmica (Últ. Século)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Desvio Padrão da Temperatura (°C)")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/home/jovyan/output/p3_cidades_instabilidade.png", dpi=150)
plt.show()

---
## Pergunta 4 — Correlação de Estações: Temp. mínima vs. máxima em zonas tropicais

In [ ]:
# =====================================================
# CÓDIGO RESPOSTA — PERGUNTA 4: Correlação tropical
# Zonas tropicais: Latitude entre -23.5° e +23.5°
# =====================================================

df_tropical = df_clean.filter(
    (F.col("Lat") >= -23.5) & (F.col("Lat") <= 23.5)
)

# Temperatura mínima e máxima mensais por cidade/ano
df_minmax = df_tropical \
    .groupBy("City", "Country", "Year") \
    .agg(
        F.min("AverageTemperature").alias("MinTemp"),
        F.max("AverageTemperature").alias("MaxTemp")
    )

# Correlação de Pearson entre MinTemp e MaxTemp
corr_result = df_minmax.stat.corr("MinTemp", "MaxTemp")
print(f"\n>>> Coeficiente de Correlação de Pearson (Tropical): {corr_result:.4f}")

if corr_result > 0.8:
    print("Interpretação: Correlação FORTE. Quando a temperatura mínima sobe, a máxima também tende a subir.")
elif corr_result > 0.5:
    print("Interpretação: Correlação MODERADA entre mínimas e máximas tropicais.")
else:
    print("Interpretação: Correlação FRACA entre mínimas e máximas tropicais.")

# Dispersão para visualização (amostra para não sobrecarregar plot)
pdf_mm = df_minmax.sample(fraction=0.05, seed=42).toPandas()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(pdf_mm["MinTemp"], pdf_mm["MaxTemp"],
           alpha=0.3, s=10, color="#27AE60")
ax.set_title(f"Correlação Temp. Min vs Max — Zonas Tropicais\nPearson r = {corr_result:.4f}",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Temperatura Mínima Anual (°C)")
ax.set_ylabel("Temperatura Máxima Anual (°C)")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/home/jovyan/output/p4_correlacao_tropical.png", dpi=150)
plt.show()

---
## Pergunta 5 — Qualidade de Dados: Registros com incerteza > 10% da média histórica

In [ ]:
# =====================================================
# CÓDIGO RESPOSTA — PERGUNTA 5: Qualidade de dados
# Usando df_raw (pré-limpeza) para análise de qualidade
# =====================================================

# Calcular a média histórica global de temperatura
global_mean = df_raw \
    .filter(F.col("AverageTemperature").isNotNull()) \
    .agg(F.avg("AverageTemperature").alias("GlobalMean")) \
    .collect()[0]["GlobalMean"]

threshold_10pct = global_mean * 0.10
print(f"Média histórica global: {global_mean:.2f}°C")
print(f"Limiar de 10% da média: {threshold_10pct:.2f}°C")

# Identificar registros problemáticos
df_quality_issues = df_raw.filter(
    F.col("AverageTemperatureUncertainty") > threshold_10pct
)

total_raw = df_raw.count()
total_issues = df_quality_issues.count()
pct_issues = (total_issues / total_raw) * 100

print(f"\nTotal de registros brutos: {total_raw:,}")
print(f"Registros com incerteza > 10% da média: {total_issues:,} ({pct_issues:.1f}%)")

# Distribuição por período
df_quality_period = df_quality_issues \
    .withColumn("Year", F.year(F.to_date(F.col("dt"), "yyyy-MM-dd"))) \
    .groupBy((F.col("Year") / 50).cast(IntegerType()) * 50) \
    .count() \
    .withColumnRenamed("(CAST((Year / 50) AS INT) * 50)", "Period") \
    .orderBy("Period")

df_quality_period.show()

---
## Pergunta 6 — Correlação CO2 vs. Aquecimento (Join entre datasets)

In [ ]:
# =====================================================
# CÓDIGO RESPOSTA — PERGUNTA 6: Join Temperatura + CO2
# =====================================================

# --- Carregar dataset de CO2 ---
df_co2_raw = spark.read.csv(
    "/home/jovyan/data/owid-co2-data.csv",
    header=True,
    inferSchema=True
)

# Limpeza CO2:
# 1. Remover agregados globais
df_co2 = df_co2_raw.filter(
    ~F.col("country").isin(["World", "Asia", "Europe", "Africa",
                             "North America", "South America", "Oceania",
                             "High-income countries", "Low-income countries"])
)

# 2. Selecionar colunas relevantes e filtrar período
df_co2 = df_co2 \
    .select(
        F.col("country").alias("CO2_Country"),
        F.col("year").cast(IntegerType()).alias("Year"),
        F.col("co2").cast(DoubleType()).alias("CO2_MtCO2")
    ) \
    .filter((F.col("Year") >= 1975) & (F.col("CO2_MtCO2").isNotNull()))

# Padronização de nomes de países para o Join
# (Os mais comuns que divergem entre os dois datasets)
df_co2 = df_co2.withColumn("CO2_Country",
    F.regexp_replace(F.col("CO2_Country"), "United States", "United States")  # já correto
).withColumn("CO2_Country",
    F.when(F.col("CO2_Country") == "United Kingdom", "United Kingdom")
    .when(F.col("CO2_Country") == "Czechia", "Czech Republic")
    .when(F.col("CO2_Country") == "South Korea", "Korea, South")
    .otherwise(F.col("CO2_Country"))
)

# --- Agregar temperatura por País/Ano ---
df_temp_country_year = df_clean \
    .filter(F.col("Year") >= 1975) \
    .groupBy("Country", "Year") \
    .agg(F.round(F.avg("AverageTemperature"), 4).alias("AvgTemp"))

# --- Join: Temperatura + CO2 por País e Ano ---
df_joined = df_temp_country_year.join(
    df_co2,
    on=[
        df_temp_country_year["Country"] == df_co2["CO2_Country"],
        df_temp_country_year["Year"] == df_co2["Year"]
    ],
    how="inner"
).select(
    df_temp_country_year["Country"],
    df_temp_country_year["Year"],
    "AvgTemp",
    "CO2_MtCO2"
)

df_joined.cache()
print(f"Registros após join: {df_joined.count():,}")
df_joined.show(10)

# Correlação de Pearson: CO2 vs Temperatura
corr_co2_temp = df_joined.stat.corr("CO2_MtCO2", "AvgTemp")
print(f"\n>>> Correlação de Pearson (CO2 vs Temperatura): {corr_co2_temp:.4f}")

# Scatter plot
pdf_joined = df_joined.sample(fraction=0.1, seed=42).toPandas()

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(pdf_joined["CO2_MtCO2"], pdf_joined["AvgTemp"],
           alpha=0.3, s=15, color="#8E44AD")
ax.set_title(f"Emissões de CO2 vs Temperatura Média por País/Ano\nPearson r = {corr_co2_temp:.4f}",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Emissões de CO2 (Mt/ano)")
ax.set_ylabel("Temperatura Média (°C)")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/home/jovyan/output/p6_co2_vs_temperatura.png", dpi=150)
plt.show()

---
## Pergunta 7 — Ranking de Aceleração Térmica (Window Functions)

In [ ]:
# =====================================================
# CÓDIGO RESPOSTA — PERGUNTA 7: Países com maior aceleração térmica
# Compara a última década (2010s) vs década anterior (2000s)
# =====================================================

df_decades_temp = df_clean \
    .filter(F.col("Decade").isin([2000, 2010])) \
    .groupBy("Country", "Decade") \
    .agg(F.round(F.avg("AverageTemperature"), 4).alias("AvgTemp"))

# Pivot: uma linha por país com temp de cada década
df_pivot = df_decades_temp.groupBy("Country").pivot("Decade").avg("AvgTemp")
df_pivot = df_pivot.withColumnRenamed("2000", "Temp_2000s") \
                   .withColumnRenamed("2010", "Temp_2010s")

# Calcular aceleração (diferença entre décadas)
df_accel = df_pivot \
    .filter(F.col("Temp_2000s").isNotNull() & F.col("Temp_2010s").isNotNull()) \
    .withColumn("TempAcceleration",
                F.round(F.col("Temp_2010s") - F.col("Temp_2000s"), 4)) \
    .orderBy(F.desc("TempAcceleration")) \
    .limit(10)

df_accel.show(truncate=False)

# Visualização
pdf_accel = df_accel.toPandas()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#E74C3C" if v > 0 else "#3498DB" for v in pdf_accel["TempAcceleration"]]
ax.barh(pdf_accel["Country"][::-1], pdf_accel["TempAcceleration"][::-1],
        color=colors[::-1], edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Top 10 Países — Maior Aceleração Térmica\n(2010s vs 2000s)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Diferença de Temperatura (°C)")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/home/jovyan/output/p7_aceleracao_termica.png", dpi=150)
plt.show()

---
## Pergunta 8 — Previsão de Tendência (Spark MLlib — Regressão Linear)

In [ ]:
# =====================================================
# CÓDIGO RESPOSTA — PERGUNTA 8: Regressão Linear com MLlib
# =====================================================

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline

# Configuração: escolha país/cidade para previsão
TARGET_COUNTRY = "Brazil"
FUTURE_YEARS = [2024, 2025, 2026, 2027, 2028]
HISTORY_YEARS = 20  # Usar últimos 20 anos de dados

# Temperatura média anual do país nos últimos 20 anos
max_year = df_clean.agg(F.max("Year")).collect()[0][0]
start_year = max_year - HISTORY_YEARS

df_ml = df_clean \
    .filter(
        (F.col("Country") == TARGET_COUNTRY) &
        (F.col("Year") >= start_year)
    ) \
    .groupBy("Year") \
    .agg(F.avg("AverageTemperature").alias("AvgTemp")) \
    .orderBy("Year")

print(f"Dados históricos — {TARGET_COUNTRY} ({start_year}–{max_year}):")
df_ml.show()

# Preparar features para MLlib
assembler = VectorAssembler(inputCols=["Year"], outputCol="features")
df_ml_vec = assembler.transform(
    df_ml.withColumn("Year", F.col("Year").cast("double"))
)

# Treinar Regressão Linear
lr = LinearRegression(featuresCol="features", labelCol="AvgTemp")
model = lr.fit(df_ml_vec)

print(f"\nCoeficiente (slope): {model.coefficients[0]:.6f} °C/ano")
print(f"Intercepto: {model.intercept:.4f}")
print(f"R²: {model.summary.r2:.4f}")

# Previsões para os próximos 5 anos
df_future = spark.createDataFrame(
    [(float(y),) for y in FUTURE_YEARS], ["Year"]
)
df_future_vec = assembler.transform(df_future)
df_predictions = model.transform(df_future_vec)

print(f"\n>>> Previsão de Temperatura — {TARGET_COUNTRY}:")
df_predictions.select(
    F.col("Year").cast(IntegerType()),
    F.round(F.col("prediction"), 2).alias("Temp_Prevista_C")
).show()

# Visualização: histórico + previsão
pdf_hist = df_ml.toPandas()
pdf_pred = df_predictions.select(
    F.col("Year").cast(IntegerType()).alias("Year"),
    F.col("prediction").alias("AvgTemp")
).toPandas()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(pdf_hist["Year"], pdf_hist["AvgTemp"], marker="o",
        color="#2980B9", label="Histórico", linewidth=2)
ax.plot(pdf_pred["Year"], pdf_pred["AvgTemp"], marker="^",
        color="#E74C3C", linestyle="--", label="Previsão MLlib", linewidth=2)
ax.axvline(max_year, color="gray", linestyle=":", alpha=0.7)
ax.set_title(f"Previsão de Temperatura — {TARGET_COUNTRY} (Regressão Linear MLlib)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Ano")
ax.set_ylabel("Temperatura Média (°C)")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/home/jovyan/output/p8_previsao_mllib.png", dpi=150)
plt.show()

---
## Saída Final — Exportar Parquet (Data Lake Simulado)

In [ ]:
# =====================================================
# ARTEFATO DE SAÍDA: Dataset final em formato Parquet
# =====================================================

# Dataset enriquecido após join com CO2
df_final = df_joined.select(
    "Country",
    "Year",
    F.round("AvgTemp", 4).alias("AvgTemperature_C"),
    F.round("CO2_MtCO2", 2).alias("CO2_Emissions_MtCO2")
)

# Salvar como Parquet particionado por Country
df_final.write \
    .mode("overwrite") \
    .partitionBy("Country") \
    .parquet("/home/jovyan/output/climate_co2_final.parquet")

print("Parquet salvo em /output/climate_co2_final.parquet")
print(f"Total de registros exportados: {df_final.count():,}")

# Dicionário de dados
print("""
=== DICIONÁRIO DE DADOS — climate_co2_final.parquet ===

Country              : Nome do país (String) — chave de partição Parquet
Year                 : Ano da medição (Integer, 1975–2023)
AvgTemperature_C     : Temperatura média anual do país em Celsius (Double)
                       Calculada como média de todas as cidades do país no ano,
                       após filtragem de incerteza < 1.5°C e nulos.
CO2_Emissions_MtCO2  : Emissões anuais de CO2 do país em megatoneladas (Double)
                       Fonte: Our World in Data (owid-co2-data.csv)
""")

In [ ]:
# Encerrar sessão Spark
spark.stop()
print("SparkSession encerrada.")